# Forecasting com SARIMAX

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | SARIMAX — Seasonal ARIMA com fatores exógenos (aplicado como univariado) |
| **Biblioteca** | `pmdarima` 2.1.1 + `statsmodels` 0.14.6 — `SARIMAX` |
| **Hiperparâmetros configurados** | `start_p=0`, `start_q=0`, `max_p=3`, `max_q=3`, `max_d=2`, `d=None`, `start_P=0`, `start_Q=0`, `max_P=2`, `max_Q=2`, `max_D=1`, `D=None`, `m=12`, `seasonal=True`, `information_criterion='aic'`, `stepwise=True` |
| **Busca de hiperparâmetros** | Sim — `auto_arima` stepwise (Hyndman-Khandakar), executado **uma vez** por série sobre o conjunto de treino |
| **Critério de seleção** | AIC (in-sample) |
| **Séries utilizadas** | 29 séries com total ≥ 48 observações (`len(series) < TEST_SIZE + 24`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses |
| **Reprodutibilidade** | Modelo determinístico — seed não aplicável |
| **Referência** | Box, G.E.P. et al. (2015). *Time Series Analysis*, 5ª ed. Wiley. Hyndman, R.J. & Khandakar, Y. (2008). *J. Stat. Softw.*, 27(3). |

---

## 1. Instalação de Dependências

In [1]:
# Instalação das dependências: pmdarima para auto_arima e statsmodels para SARIMAX
%pip install pmdarima statsmodels -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importação de Bibliotecas

## 3. Carregamento dos Dados

In [2]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from pmdarima import auto_arima                         # Seleção automática de (p,d,q)(P,D,Q,m)
from statsmodels.tsa.statespace.sarimax import SARIMAX  # Modelo SARIMAX (statsmodels)

warnings.filterwarnings('ignore')

# Carregar a base de dados com 29 séries econômicas brasileiras (108 meses cada)
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)
print(f"Shape: {df.shape} | Período: {df.index.min():%Y-%m} a {df.index.max():%Y-%m}")
print(f"Séries: {', '.join(df.columns)}")

Shape: (108, 29) | Período: 2017-01 a 2025-12
Séries: IBC_Br, Selic, Cambio_USDBRL, Desemprego, Brent_USD, Soja_USD, Minerio_USD, Ibovespa, ICC_FGV, Credito_Total, Inadimplencia, Massa_Salarial, CPI_USA, Prod_Ind_USA, Cafe_USD, Ouro_USD, GasNatural_USD, Cobre_USD, ETF_Emergentes, INCC, ICE_Empresarial, Housing_Starts_EUA, Dollar_Index_Fed, PMS_Volume, M2, Divida_PIB, Vendas_Varejo, NUCI_FGV, SP500


## 4. Configuração do Experimento

In [3]:
# ============================================================
# Constantes do experimento
# ============================================================
TEST_SIZE = 24          # Período de teste: últimos 24 meses da base
HORIZON = 3             # Horizonte de previsão: 3 meses à frente
SEASONAL_PERIOD = 12    # Sazonalidade: 12 meses (dados mensais)


def find_best_order(series):
    """
    Usa auto_arima para encontrar automaticamente os melhores parâmetros SARIMAX.
    
    Busca no espaço:
      - (p,d,q): componentes não-sazonais (AR, diferenciação, MA)
        p ∈ [0,3], d auto-detectado, q ∈ [0,3]
      - (P,D,Q,m): componentes sazonais
        P ∈ [0,2], D auto-detectado, Q ∈ [0,2], m=12
    
    A seleção minimiza o AIC (Akaike Information Criterion),
    que balanceia qualidade de ajuste vs. complexidade do modelo.
    
    Returns:
        order: tupla (p, d, q)
        seasonal_order: tupla (P, D, Q, 12)
    """
    model = auto_arima(
        series,
        start_p=0, start_q=0,
        max_p=3, max_q=3, max_d=2,
        d=None,                      # Auto-detectar diferenciação
        start_P=0, start_Q=0,
        max_P=2, max_Q=2, max_D=1,
        D=None,                      # Auto-detectar diferenciação sazonal
        m=SEASONAL_PERIOD,
        seasonal=True,
        information_criterion='aic',
        stepwise=True,               # Algoritmo stepwise (rápido)
        error_action='ignore',
        suppress_warnings=True,
        trace=False,
    )
    return model.order, model.seasonal_order


def calc_metrics(y_true, y_pred):
    """Calcula MAE, RMSE e MAPE. Usa epsilon 1e-8 para evitar divisão por zero no MAPE."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}


def classificar(mape):
    """Classifica desempenho de previsão por faixa de MAPE."""
    if mape < 10:
        return '⭐ Excelente'
    elif mape < 20:
        return '✅ Muito Bom'
    elif mape < 30:
        return '👍 Bom'
    elif mape < 50:
        return '⚠️ Regular'
    else:
        return '❌ Difícil'


print(f"Configuração: TEST_SIZE={TEST_SIZE}, HORIZON={HORIZON}, SEASONAL_PERIOD={SEASONAL_PERIOD}")
print(f"auto_arima: p∈[0,3], q∈[0,3], P∈[0,2], Q∈[0,2], critério=AIC")
print(f"Walk-forward: multi-step ({HORIZON} meses) com retreino a cada janela")

Configuração: TEST_SIZE=24, HORIZON=3, SEASONAL_PERIOD=12
auto_arima: p∈[0,3], q∈[0,3], P∈[0,2], Q∈[0,2], critério=AIC
Walk-forward: multi-step (3 meses) com retreino a cada janela


In [4]:
print("=" * 70)
print(f"SARIMAX — Seleção por AIC + Walk-Forward Multi-Step (horizon={HORIZON})")
print("=" * 70)

ALL_SERIES = list(df.columns)  # Lista com as 29 séries econômicas
# Excluir PIM e IPCA_mensal da análise
ALL_SERIES = [s for s in ALL_SERIES if s not in ['PIM', 'IPCA_mensal']]
all_results = []                # Armazena métricas e previsões de cada série
selected_orders = {}            # Guarda os parâmetros selecionados por série

for idx, col in enumerate(ALL_SERIES, 1):
    series = df[col].dropna()

    # Verificação de segurança: série precisa ter dados suficientes
    if len(series) < TEST_SIZE + 24:
        print(f"[{idx}/{len(ALL_SERIES)}] {col} — série muito curta, pulando")
        continue

    # Separação temporal: treino (84 meses) e teste (24 meses finais)
    train_s = series.iloc[:-TEST_SIZE]
    test_s = series.iloc[-TEST_SIZE:]

    # ============================================================
    # FASE 1: Seleção de parâmetros via auto_arima (no treino apenas)
    # Encontra (p,d,q)(P,D,Q,12) que minimiza AIC
    # ============================================================
    try:
        order, seasonal_order = find_best_order(train_s.values)
        selected_orders[col] = (order, seasonal_order)
    except Exception as e:
        print(f"[{idx:2}/{len(ALL_SERIES)}] {col:20} | auto_arima falhou: {e}")
        continue

    # ============================================================
    # FASE 2: Walk-Forward multi-step (8 janelas de 3 meses)
    # A cada janela, retreina SARIMAX com janela expansível e prevê 3 meses
    # ============================================================
    forecasts, actuals = [], []

    for i in range(0, len(test_s), HORIZON):
        # Número de passos nesta janela (última pode ser menor que HORIZON)
        n_steps = min(HORIZON, len(test_s) - i)

        # Janela expansível: treino original + pontos de teste já revelados
        if i == 0:
            history = train_s.values
        else:
            history = np.concatenate([train_s.values, test_s.values[:i]])

        try:
            model = SARIMAX(history,
                            order=order,
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False,
                            enforce_invertibility=False)
            fitted = model.fit(disp=False, maxiter=200)
            fc = fitted.forecast(steps=n_steps)
        except Exception:
            fc = np.full(n_steps, history[-HORIZON:].mean())

        forecasts.extend(fc)
        actuals.extend(test_s.values[i:i + n_steps])

    forecasts = np.array(forecasts)
    actuals = np.array(actuals)

    # Cálculo das métricas
    metrics = calc_metrics(actuals, forecasts)
    classif = classificar(metrics['MAPE'])

    all_results.append({
        'Serie': col,
        'MAE': round(metrics['MAE'], 4),
        'RMSE': round(metrics['RMSE'], 4),
        'MAPE': round(metrics['MAPE'], 2),
        'N_Pontos': len(forecasts),
        'Order': str(order),
        'Seasonal_Order': str(seasonal_order),
        'Classificacao': classif,
        'actuals': actuals,
        'preds': forecasts,
    })
    print(f"[{idx:2}/{len(ALL_SERIES)}] {col:20} | MAPE={metrics['MAPE']:7.2f}%  MAE={metrics['MAE']:.4f}  {order} x {seasonal_order}  {classif}")

print(f"\n{'='*60}")
print(f"✅ Walk-forward concluído para {len(all_results)} séries")

SyntaxError: incomplete input (2148052350.py, line 47)

## 5. Resultados e Métricas

In [ ]:
# Monta DataFrame de resultados (sem arrays de previsões) ordenado por MAPE
df_res = pd.DataFrame([{k: v for k, v in r.items() if k not in ('preds', 'actuals')}
                       for r in all_results]).sort_values('MAPE')

print("=" * 60)
print("RESULTADOS — SARIMAX (auto_arima + Walk-Forward)")
print("=" * 60)
print(f"\nMAPE médio: {df_res['MAPE'].mean():.2f}%  |  MAE médio: {df_res['MAE'].mean():.4f}  |  RMSE médio: {df_res['RMSE'].mean():.4f}\n")

# Ranking com medalhas para as 3 melhores séries
for i, (_, r) in enumerate(df_res.iterrows(), 1):
    tag = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{tag} {i:2}. {r['Serie']:20} MAPE={r['MAPE']:7.2f}%  MAE={r['MAE']:.4f}  {r['Order']} x {r['Seasonal_Order']}")

RESULTADOS — SARIMAX (auto_arima + Walk-Forward)

MAPE médio: 8623115.54%  |  MAE médio: 4038391.5615  |  RMSE médio: 4977446.1907

🥇  1. CPI_USA              MAPE=   0.12%  MAE=0.4016  (1, 2, 2) x (0, 0, 0, 12)
🥈  2. Prod_Ind_USA         MAPE=   0.42%  MAE=0.4242  (3, 0, 0) x (0, 0, 0, 12)
🥉  3. Divida_PIB           MAPE=   0.83%  MAE=0.5314  (1, 1, 0) x (0, 0, 1, 12)
    4. Massa_Salarial       MAPE=   0.96%  MAE=3482.5000  (0, 1, 0) x (0, 0, 0, 12)
    5. EAI_Emprego_Ind      MAPE=   0.98%  MAE=31744.4326  (1, 1, 0) x (2, 1, 0, 12)
    6. M2                   MAPE=   1.00%  MAE=141250565.5000  (0, 2, 0) x (0, 0, 0, 12)
    7. IBC_Br               MAPE=   1.08%  MAE=1.1888  (2, 1, 2) x (2, 0, 0, 12)
    8. Credito_Total        MAPE=   1.25%  MAE=48057.5833  (0, 2, 0) x (0, 0, 0, 12)
    9. NUCI_FGV             MAPE=   1.42%  MAE=1.1607  (2, 1, 1) x (1, 0, 0, 12)
   10. PMS_Volume           MAPE=   1.45%  MAE=1.5833  (1, 1, 2) x (1, 0, 1, 12)
   11. Selic                MAPE=   1.59% 

## 6. Visualização: Ranking MAPE por Série

## 7. Exportação de Resultados

In [ ]:
# ── Exportação dos Resultados ──# DataFrame de métricas (sem arrays)results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('preds', 'actuals')}                           for r in all_results]).sort_values('MAPE')results_df.to_csv('resultados_sarimax.csv', index=False)print(f"✅ Salvo: resultados_sarimax.csv ({len(results_df)} séries)")# DataFrame de previsões detalhadasrows = []for r in all_results:    for j, (real, pred) in enumerate(zip(r['actuals'], r['preds'])):        rows.append({'Serie': r['Serie'], 'Step': j+1, 'Real': real, 'Previsto': pred})df_prev = pd.DataFrame(rows)df_prev.to_csv('previsoes_sarimax.csv', index=False)print(f"✅ Salvo: previsoes_sarimax.csv ({len(df_prev)} linhas)")

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = df_res.sort_values('MAPE')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df['Serie'])
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'SARIMAX — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('sarimax_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: sarimax_mape_por_serie.png")

## 8. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df['Serie'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    r = next(x for x in all_results if x['Serie'] == sn)

    # Valores reais (teste)
    ax.plot(r['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(r['preds'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {r['MAPE']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('SARIMAX — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('sarimax_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: sarimax_previsoes.png")